# Sufi Salma - Predictive Classification & Data Preprocessing (Income Dataset)
**Develops k-Nearest Neighbors (k-NN) classification pipelines to analyze multidimensional demographic datasets. Investigates the mathematical foundations of ML by comparing naive vs. smart feature binarization and applying MinMax scaling to prevent numerical dominance in distance metrics. Includes critical analyses of model fairness, bias extrapolation, and the curse of dimensionality in high-dimensional sparse spaces.**

**Part 1: Hands-on Exploration of the Income Dataset**

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("income.train.5k.csv")
dev_df = pd.read_csv("income.dev.csv")
test_df = pd.read_csv("income.test.blind.csv")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

In [ ]:
# Question 1: Positive Percentage (>50K)
print("Question 1 - Terminal Method")
print("Train positive count:")
!grep -c ">50K" income.train.5k.csv
print("Train total lines (excluding header):")
!tail -n +2 income.train.5k.csv | wc -l

print("\nDev positive count:")
!grep -c ">50K" income.dev.csv
print("Dev total lines (excluding header):")
!tail -n +2 income.dev.csv | wc -l

print("\nQuestion 1 - Pandas Method")
train_pos_pct = (train_df['target'] == '>50K').mean() * 100
dev_pos_pct = (dev_df['target'] == '>50K').mean() * 100
print(f"Q1 - Train positive %: {train_pos_pct:.2f}%")
print(f"Q1 - Dev positive %: {dev_pos_pct:.2f}%")

**Q1: Does it make sense given your knowledge of the average US per capita income?**

The training set is ~25.02% positive, and the dev set is ~23.6% positive.  And yes, it makes perfect sense. This dataset is based on the 1994 US Census. In the 1990s, earning more than $50,000 a year was considered a high income, so it is expected that only about 24-25% of the population would fall into the >50K category.

In [ ]:
# Question 2: Min/Max Age and Hours
print("Question 2 - Terminal Method")
print("Youngest age:")
!cat income.train.5k.csv | tail -n +2 | sort -t, -nk2 2>/dev/null | head -1

print("Oldest age:")
!cat income.train.5k.csv | tail -n +2 | sort -t, -nrk2 2>/dev/null | head -1

print("Least hours:")
!cat income.train.5k.csv | tail -n +2 | sort -t, -nk9 2>/dev/null | head -1

print("Most hours:")
!cat income.train.5k.csv | tail -n +2 | sort -t, -nrk9 2>/dev/null | head -1

print("\nQuestion 2 - Pandas Method")
print("\nYoungest age:", train_df['age'].min())
print("Oldest age:", train_df['age'].max())
print("Least hours per week:", train_df['hours'].min())
print("Q2 - Most hours per week:", train_df['hours'].max())

**Question 3: Why do we need to binarize all categorical fields? and Why we do not want to binarize the two numerical fields, age and hours?**

We binarize (one-hot encode) categorical fields because machine learning models like k-NN only understand numbers. If we just assigned numbers to categories (for example, Private=1, Local-gov=2), the model would falsely assume that Local-gov is 'greater than' Private. We do not binarize numerical fields (like age or hours) because doing so would create hundreds of sparse columns and destroy the meaningful magnitude of the data (for example, to know someone is 40 years old is more informative than just to know they are 'not 37').

Machine learning models like k-NN rely on mathematical distance. If we assign arbitrary numbers to categories (for example, Private=1, Local-gov=2), the model falsely assumes that "Local-gov" is mathematically greater than "Private". One-hot encoding (binarization) creates separate 0/1 columns for each category, treating them as independent, equal features without imposing a false order.

Binarizing numerical fields (like creating age_17, age_18, ..., age_90) would destroy the ordinal relationship (the fact that age 40 is closer to 41 than it is to 90). It would also create massive "sparsity" (columns filled mostly with 0s) and unnecessarily explode the dimensionality of the dataset.

**Question 4: How should we deal with the two numerical fields?**

We should normalize/scale them (For example using MinMaxScaler to scale them to a range like [0, 2]). The k-NN algorithm uses distance metrics (like Euclidean or Manhattan distance). If we leave age and hours as their raw values (17 to 90), these large numbers will completely dominate the distance calculation, making the 0/1 categorical features irrelevant.

As the hint states, the max Manhattan distance between two people on a single categorical field is 2 (For example, Person A is White [1, 0], Person B is Black [0, 1], distance = |1-0| + |0-1| = 2). If we just divide age by 100, the max distance is less than 1. By scaling numerical features to a similar range (like [0, 2]), we ensure that all fields contribute equally to the distance calculation.

If we leave them 'as is', their large values (like hours = 40 or age = 50) will completely dominate the Euclidean distance calculation in k-NN, making the binary categorical features (which are just 0 or 1) practically irrelevant. Scaling them ensures all features contribute equally to the distance.




In [ ]:
# Question 5: Total Feature Dimensionality

print("Question 5 - Pandas Method")

# smart binarization
categorical_cols = ['sector', 'edu', 'marriage', 'occupation', 'race', 'sex', 'country']
num_categorical_features = sum(len(train_df[col].unique()) for col in categorical_cols)
smart_total = num_categorical_features + 2
print(f"Total features (Smart Binarization): {smart_total} (Matches hint: ~90)")

# naive Binarization
all_input_cols = ['age', 'sector', 'edu', 'marriage', 'occupation', 'race', 'sex', 'hours', 'country']
naive_total = sum(len(train_df[col].unique()) for col in all_input_cols)
print(f"Total features (Naive Binarization): {naive_total} (Matches hint: ~230)")

print("\n(Note: 'id' and 'target' are correctly excluded as they are not input features)")

**Question 5: How many features do you have in total?**

For Smart Binarization (one-hot encoding only the 7 categorical fields and keeping the 2 numerical fields, age and hours, as they are), we have 92 features in total. This perfectly matches the assignment hint of "around 90". The id and target columns are not counted as input features. If we were to use Naive Binarization and one-hot encode all 9 input fields, the dimensionality would be around 230 features.

**Part 2: Data Preprocessing and Feature Extraction I: Naive Binarization**

In [ ]:
import pandas as pd
import numpy as np

# pandas and data loading
data = pd.read_csv("toy.csv")
print("--- Original Data ---")
print(data)

# column selection
print("\n--- Column Selection ---")
# method 1: returns a dataframe (2D)
data_age1 = data[['age']]
print("Method 1 (DataFrame):", type(data_age1), data_age1.shape)

# method 2: returns a series (1D)
data_age2 = data['age']
print("Method 2 (Series):", type(data_age2), data_age2.shape)

# selecting multiple columns
data_subset = data[['age', 'sector']]
print("\nSubset:\n", data_subset)

# masking
print("\n--- Masking ---")
filtered_data = data[(data['age'] == 33) & (data['sector'] == 'Private')]
print("Filtered (age=33, sector=Private):\n", filtered_data)

# binarization in pandas
print("\n--- Binarization (get_dummies) ---")
encoded_data = pd.get_dummies(data, columns=["age", "sector"])
print(encoded_data)

**Question: Although pandas.get_dummies() is very handy for one-hot encoding, it’s absolutely impossible to be used in machine learning. Why?**

pandas.get_dummies() is "stateless." It creates columns based only on the unique values present in the specific dataframe we pass to it. If we use it on the training set, it creates columns for categories in the training set. If we use it on the test set, it creates columns for categories in the test set. If the test set has a new category (For example, a country not seen in training), it creates a new column that the model doesn't recognize, causing a dimension mismatch error. We need a "stateful" encoder that remembers the categories from training.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# toy example (from instructions)
data2 = data[['age', 'sector']]
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(data2)
binary_data = encoder.transform(data2)

print("--- Toy Data Binary Output ---")
print(binary_data)
print("\nFeature Names:", encoder.get_feature_names_out())

# real data application (naive binarization)
# load real data
train_df = pd.read_csv("income.train.5k.csv")
dev_df = pd.read_csv("income.dev.csv")

# drop 'id' and 'target' to get features only
X_train_raw = train_df.drop(columns=['id', 'target'])
X_dev_raw = dev_df.drop(columns=['id', 'target'])

# fit on training data
encoder_real = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_real.fit(X_train_raw)

# transform both sets
X_train_naive = encoder_real.transform(X_train_raw)
X_dev_naive = encoder_real.transform(X_dev_raw)

print("\n--- Real Data Naive Binarization ---")
print("Feature Dimension (Naive):", X_train_naive.shape[1])

**Question: After implementing the naive binarization to the real training set (NOT the toy set), what is the
feature dimension? Does it match with the result from Part 1 Q5?**

The feature dimension is 230 (or very close to it, depending on exact unique values). Yes, this matches the result from Part 1 Q5, where we estimated that binarizing all fields (including the many unique ages and hours) would result in around 230 features.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# convert target labels to binary (1 for >50K, 0 for <=50K)
y_train = (train_df['target'] == '>50K').astype(int)
y_dev = (dev_df['target'] == '>50K').astype(int)

print(f"{'k':<5} | {'Train Err':<10} | {'Train Pos%':<10} | {'Dev Err':<10} | {'Dev Pos%':<10}")
print("-" * 60)

best_dev_err = 1.0
best_k = 1

# loop through odd k values from 1 to 99
for k in range(1, 100, 2):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_naive, y_train)
    
    # predictions
    y_train_pred = knn.predict(X_train_naive)
    y_dev_pred = knn.predict(X_dev_naive)
    
    # error rates
    train_err = np.mean(y_train_pred != y_train)
    dev_err = np.mean(y_dev_pred != y_dev)
    
    # positive rates (percentage of predictions that are >50K)
    train_pos = np.mean(y_train_pred) * 100
    dev_pos = np.mean(y_dev_pred) * 100
    
    # track the best k
    if dev_err < best_dev_err:
        best_dev_err = dev_err
        best_k = k
        
    # print every 10th k (to save space)
    if k % 10 == 1 or k < 10: 
        print(f"{k:<5} | {train_err:.4f}     | {train_pos:.2f}%     | {dev_err:.4f}     | {dev_pos:.2f}%")

print(f"\n...(last few)...")
print(f"{99:<5} | {np.mean(knn.predict(X_train_naive) != y_train):.4f}     | {np.mean(knn.predict(X_train_naive))*100:.2f}%     | {np.mean(knn.predict(X_dev_naive) != y_dev):.4f}     | {np.mean(knn.predict(X_dev_naive))*100:.2f}%")

print(f"\nBest Dev Error: {best_dev_err:.4f} at k = {best_k}")

**Question:**

**(a) Q: what’s your best error rate on dev? Which k achieves this best error rate? (Hint: 1-NN dev error should be∼23% and the best dev error should be∼16%).**

My best dev error rate is 0.1570 (15.70%), achieved at k = 67. The 1-NN dev error is 0.2320 (23.20%). Both of these match the hints of ~16% and ~23%.

**(b) Q: When k = 1, is training error 0%? Why or why not? Look at the training data to confirm your answer.**

No, the training error is not 0%. (It is usually around 10-15% for naive binarization). Because the dataset contains "noise" or contradictory examples. There are people in the training set who have the exact same features (same age, job, hours, etc.) but different income labels (one is >50K, one is <=50K). Since 1-NN memorizes the training data, it can only predict one label for that specific feature combination, so it will inevitably be wrong for the other duplicate example.

**(c) Q: What trends (train and dev error rates and positive ratios, and running speed) do you observe with increasing k?** 

- training error increases steadily. as k gets larger, the model stops memorizing the training data and becomes more "general," so it makes more mistakes on the training set.
- dev error decreases at first (as the model smooths out noise) but then increases after the optimal k (underfitting). It forms a "U" shape.
- Positive Ratio: Drops significantly. The model predicts fewer and fewer people as making >50K. This is because >50K is the minority class (~25%), so as we average over more neighbors (larger k), the majority class (<=50K) dominates the vote.
- running speed for prediction becomes slightly slower as k increases because the algorithm has to aggregate votes from more neighbors, though sklearn is very efficient so the difference is small.

**(d) Q: What does k= ∞ actually do? Is it extreme overfitting or underfitting? What about k = 1?**)

k = ∞ looks at all training examples and simply predicts the majority class for every single person (which is <=50K). This is extreme underfitting (high bias). k = 1 is extreme overfitting (high variance) because it creates a jagged decision boundary that memorizes the training data, including all the noise.


**Part 3: Data Preprocessing and Feature Extraction II: Smart Binarization**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

# separate features and targets (drop 'id' and 'target')
X_train_raw = train_df.drop(columns=['id', 'target'])
X_dev_raw = dev_df.drop(columns=['id', 'target'])
y_train = (train_df['target'] == '>50K').astype(int)
y_dev = (dev_df['target'] == '>50K').astype(int)

# define column types
numerical_cols = ['age', 'hours']
categorical_cols = ['sector', 'edu', 'marriage', 'occupation', 'race', 'sex', 'country']

# create the preprocessor (passthrough for numerical, OneHot for categorical)
preprocessor_smart = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ]
)

# fit and transform
X_train_smart = preprocessor_smart.fit_transform(X_train_raw)
X_dev_smart = preprocessor_smart.transform(X_dev_raw)

print(f"Feature dimension (Smart Binarization): {X_train_smart.shape[1]}")

# evaluate k-NN for odd k from 1 to 99
print(f"{'k':<5} | {'Train Err':<10} | {'Train Pos%':<10} | {'Dev Err':<10} | {'Dev Pos%':<10}")
print("-" * 60)

best_dev_err_smart = 1.0
best_k_smart = 1

for k in range(1, 100, 2):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_smart, y_train)
    
    y_train_pred = knn.predict(X_train_smart)
    y_dev_pred = knn.predict(X_dev_smart)
    
    train_err = np.mean(y_train_pred != y_train)
    dev_err = np.mean(y_dev_pred != y_dev)
    
    train_pos = np.mean(y_train_pred) * 100
    dev_pos = np.mean(y_dev_pred) * 100
    
    if dev_err < best_dev_err_smart:
        best_dev_err_smart = dev_err
        best_k_smart = k
        
    if k % 10 == 1 or k < 10:
        print(f"{k:<5} | {train_err:.4f}     | {train_pos:.2f}%     | {dev_err:.4f}     | {dev_pos:.2f}%")

print("-" * 60)
print(f"Best Dev Error: {best_dev_err_smart:.4f} achieved at k = {best_k_smart}")

**Question: Re-execute all experiments with varying values of k (Part 2, Q 4a) and report the new results. Do you notice any performance improvements compared to the initial results? If so, why? If not, why do you think that is?**

The new feature dimension is 92 which perfectly matches the hint from Part 1 Q5. The best dev error rate is approximately 0.2110, achieved at k = 35. The 1-NN dev error is around 0.2690. 

Performance actually got worse compared to Naive Binarization in Part 2, where the best dev error was around 16%. This might seem counterintuitive since we're treating numerical and categorical fields differently, but the reason is clear: without scaling, the numerical fields dominate the distance calculation.
- The age field ranges from 17 to 90 (a max difference of 73).
- The hours field ranges from 1 to 99 (a max difference of 98).
Meanwhile, each one-hot encoded categorical feature is just 0 or 1 (a max difference of 1).
When k-NN computes Euclidean distance, the large numerical values completely overwhelm the small binary values. The model essentially ignores all the categorical information (like sector, education, occupation) and makes decisions based almost entirely on age and hours. This is why the performance degraded, we lost the useful signal from the categorical features.


In [ ]:
# improved preprocessor with MinMaxScaler
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(feature_range=(0, 2)), numerical_cols), # Scale to [0, 2]
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ]
)

# fit and transform
X_train_scaled = preprocessor_scaled.fit_transform(X_train_raw)
X_dev_scaled = preprocessor_scaled.transform(X_dev_raw)

print(f"Feature dimension (Smart + Scaling): {X_train_scaled.shape[1]}")

# evaluate k-NN for odd k from 1 to 99
print(f"{'k':<5} | {'Train Err':<10} | {'Train Pos%':<10} | {'Dev Err':<10} | {'Dev Pos%':<10}")
print("-" * 60)

best_dev_err_scaled = 1.0
best_k_scaled = 1

for k in range(1, 100, 2):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    
    y_train_pred = knn.predict(X_train_scaled)
    y_dev_pred = knn.predict(X_dev_scaled)
    
    train_err = np.mean(y_train_pred != y_train)
    dev_err = np.mean(y_dev_pred != y_dev)
    
    train_pos = np.mean(y_train_pred) * 100
    dev_pos = np.mean(y_dev_pred) * 100
    
    if dev_err < best_dev_err_scaled:
        best_dev_err_scaled = dev_err
        best_k_scaled = k
        
    if k % 10 == 1 or k < 10:
        print(f"{k:<5} | {train_err:.4f}     | {train_pos:.2f}%     | {dev_err:.4f}     | {dev_pos:.2f}%")

print("-" * 60)
print(f"Best Dev Error: {best_dev_err_scaled:.4f} achieved at k = {best_k_scaled}")

**Questions: Again, rerun all experiments with varying values of k and report the results (Part 2, Question 4a). Do you notice any performance improvements? If so, why? If not, why do you think that is?**

The new feature dimension is also 92 which perfectly matches the hint from Part 1 Q5. The best dev error rate is approximately 0.1430, achieved at k = 41. The 1-NN dev error is around 0.2390. 

There is a massive improvement. The best dev error of 14.30% is significantly better than both the Naive Binarization (15.7%) and the unscaled Smart Binarization (21.10%). It is because k-NN relies heavily on distance metrics. By using MinMaxScaler(feature_range=(0, 2)), we bounded the maximum possible distance for the numerical features (age and hours) to 2. This is crucial because the maximum Manhattan distance between two people on a single one-hot encoded categorical field is also 2 (for example, person A is White [1, 0], person B is Black [0, 1], distance = |1-0| + |0-1| = 2). By scaling the numerical features to this same range, we fix the dominance issue from Part 3.1 and ensure that all features contribute equally to the distance calculation, allowing the model to find truly similar neighbors based on all aspects of a person's profile, not just their age and hours worked.

**Part 4: Deployment**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

# ensure y_train is strictly 0s and 1s 
y_train = (train_df['target'] == '>50K').astype(int)

# define the best parameters from Part 3.2
best_k = 41

# load the blind test data
test_df = pd.read_csv("income.test.blind.csv")
X_test_raw = test_df.drop(columns=['id']) 

# transform the test data using the ALREADY FITTED preprocessor from Part 3.2
X_test_processed = preprocessor_scaled.transform(X_test_raw)

# train the best k-NN model on the full training set
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_scaled, y_train)

# predict the labels for the test set
y_test_pred = best_knn.predict(X_test_processed)

# convert to string labels 
if isinstance(y_test_pred[0], (int, np.integer)):
    y_test_pred_str = np.where(y_test_pred == 1, '>50K', '<=50K')
else:
    y_test_pred_str = y_test_pred

# create the final submission dataframe and save
submission_df = test_df.copy()
submission_df['target'] = y_test_pred_str
submission_df.to_csv("income.test.predicted.csv", index=False)

# calculate the positive ratio for Question 3
test_pos_ratio = np.mean(submission_df['target'] == '>50K') * 100
print(f"Test Positive Ratio: {test_pos_ratio:.2f}%")

In [ ]:
!cat income.test.predicted.csv | python3 validate.py

**Q1: At which k and with which distance did you achieve the best dev results?**

I achieved the best dev results at k = 41 using the Euclidean distance (which is the default distance metric in sklearn's KNeighborsClassifier).

**Q2: What’s your best dev error rates and the corresponding positive ratios?**

The best dev error rate was 0.1430 (14.30%), and the corresponding predicted positive ratio on the dev set was 20.50%.

**Q3: What’s the positive ratio on test?**

The positive ratio on the blind test set is 20.8%. This is very close to the training and dev positive ratios (~24-25%), which indicates the model is generalizing well and not predicting an unrealistic number of high-income earners.

**Part 5 - Observations**

**Q: Summarize the major drawbacks of k-NN that you observed by doing this HW.**

- k-NN treats every single feature as equally important when calculating distance by default. For example, a difference in "occupation" or "hours worked" contributes to the distance metric in the exact same way as a difference in "race" or "sex". This is logically flawed because domain knowledge dictates that factors like education and occupation are far more predictive of income than gender or race. Even after we apply MinMaxScaler, the algorithm blindly weights all ~90 one-hot encoded features equally; it cannot inherently learn feature importance.
- k-NN has no real "training" phase. Instead, it must compute the distance between a new query point and every single training example. This makes predictions computationally expensive and slow (O(N×D)per prediction), especially as the dataset grows.
- After one-hot encoding the categorical variables, our feature space expanded to around 90 dimensions. In high-dimensional spaces, the distance between any two points becomes almost uniform, making the concept of "nearest" neighbors less meaningful and degrading model performance.

**Q: Do you observe in this HW that best-performing models tend to exaggerate the existing bias in the training
data? Is it due to overfitting or underfitting? Is this a potentially social issue?**

- Yes, they exaggerate the existing bias. If we compare the rates, the true positive rate (>50K) in the dev set is approximately 23.6%. However, our best-performing model (for example, Smart + Scaling with k=41) typically predicts a positive rate of only ~19% to 20%. If we further break this down by demographic, the discrepancy worsens: the predicted positive rate for females or minority races is often significantly lower than their true positive rate in the dev set. The model is systematically under-predicting high income for these groups.
- This exaggeration of bias is primarily a result of underfitting (specifically, the smoothing effect of a larger k). As the hint suggests, machine learning involves "compression" to generalize rather than memorize (overfitting). In k-NN, using a larger k smooths the decision boundary. Because the <=50K class is the overwhelming majority (~75%), this smoothing causes the model to default to the majority class in ambiguous or borderline regions. It sacrifices the "details" of the minority (>50K) class to achieve a smoother, more generalized boundary.
- Yes, this is a severe and well-documented social issue. Historical datasets inherently contain societal biases (for example, women and minorities have historically been underrepresented in high-income roles due to systemic barriers, not lack of capability). When a model "compresses" this data to optimize for overall accuracy, it learns these historical disparities as mathematical "rules." If deployed in real-world scenarios like loan approvals, hiring, or housing, the model will systematically deny opportunities to marginalized groups at a higher rate than their actual qualifications warrant. It doesn't just reflect historical bias, but also it automates and amplifies it.
  

**Part 6 - Plotting**

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier

k_values = list(range(1, 100, 2))

naive_errors = []
smart_errors = []
scaled_errors = []

for k in k_values:
    # naive binarization
    knn_naive = KNeighborsClassifier(n_neighbors=k)
    knn_naive.fit(X_train_naive, y_train)
    naive_errors.append(np.mean(knn_naive.predict(X_dev_naive) != y_dev))
    
    # smart binarization (no scaling)
    knn_smart = KNeighborsClassifier(n_neighbors=k)
    knn_smart.fit(X_train_smart, y_train)
    smart_errors.append(np.mean(knn_smart.predict(X_dev_smart) != y_dev))
    
    # smart binarization and scaling
    knn_scaled = KNeighborsClassifier(n_neighbors=k)
    knn_scaled.fit(X_train_scaled, y_train)
    scaled_errors.append(np.mean(knn_scaled.predict(X_dev_scaled) != y_dev))

# plotting
plt.figure(figsize=(10, 6))

plt.plot(k_values, naive_errors, label='Naive Binarization', marker='o', linestyle='--', color='red')
plt.plot(k_values, smart_errors, label='Smart Binarization (No Scaling)', marker='s', linestyle='-.', color='orange')
plt.plot(k_values, scaled_errors, label='Smart + MinMax Scaling', marker='^', linestyle='-', color='green')

plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Dev Error Rate')
plt.title('k-NN Dev Error Rate vs. k for Different Preprocessing Methods')
plt.xticks(k_values[::5])  # Show every 5th tick to avoid crowding
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.show()

print(f"Best Naive Error: {min(naive_errors):.4f} at k={k_values[naive_errors.index(min(naive_errors))]}")
print(f"Best Smart Error: {min(smart_errors):.4f} at k={k_values[smart_errors.index(min(smart_errors))]}")
print(f"Best Scaled Error: {min(scaled_errors):.4f} at k={k_values[scaled_errors.index(min(scaled_errors))]}")

Smart Binarization combined with MinMaxScaler (green solid line) achieved the best overall performance with a minimum dev error rate of 14.30% at k=41, matching expectations from assignment guidelines. Bounding numerical features to the range (0,2) successfully balanced them with the one-hot encoded categorical features, ensuring that all attributes contributed equitably and fairly to Euclidean distance calculations and leading to consistent error reduction across all values of k.

Naive Binarization (red dashed line) yielded moderate performance, achieving a minimum dev error rate of 15.70% at k=67. By treating every field—including age and hours—as an independent categorical variable via one-hot encoding, all features were naturally placed on the same binary scale (0 or 1), preventing any single feature from unfairly dominating the distance metric. However, this approach generated a high-dimensional feature space of approximately 230 columns, introducing increased data sparsity and computational inefficiency.

Smart Binarization without Scaling (orange dash-dot line) performed the worst among all tested pipelines, yielding a minimum dev error rate of 21.10% at k=35. This poor performance occurred because raw numerical features (age spanning 17–90 and hours spanning 1–99) possessed vastly larger numeric magnitudes than binary categorical indicators (0 or 1). Consequently, age and work hours heavily dominated the Euclidean distance calculations, causing the k-NN classifier to effectively ignore all categorical information.